Para começar:

In [ ]:
%pip install requests pandas
import requests
import base64
import urllib.parse
import pandas as pd
import time
import math

Note: you may need to restart the kernel to use updated packages.


Tentativa - De 2026 extrair patentes do European Patent Office (EPO) para um csv, mais precisanente os metadados. As patentes são só extraídas que tiverem um conjunto de palavras nos metados como no resumo, no título ou nas revindicações.

In [ ]:
import time
import base64
import urllib.parse
import requests
import pandas as pd

# ==========================================
# 1. CONFIGURAÇÕES GERAIS E CREDENCIAIS
# ==========================================

# Credenciais da API Open Patent Services (OPS) do EPO
# Coloca as tuas chaves originais dentro das aspas abaixo
CONSUMER_KEY = "" 
CONSUMER_SECRET = ""

# Parâmetros da pesquisa
# Lista otimizada com wildcards (*) para capturar singulares, plurais e variações (ex: interfer* apanha interfering e interference)
termos_rna = [
    "siRNA*", "SiRNA*", "iRNA*", "IRNA*", 
    "RNAi*", "RNAI*", 
    "small interfer* RNA*", "short interfer* RNA*", 
    "RNA interfer*", 
    "RNA interfer* agent*", "RNAi agent*", 
    "double-stranded ribonucleic acid*", "double stranded ribonucleic acid*", 
    "double stranded RNA*", "double stranded RNAi*", 
    "dsRNA*", "dsRNAi*",
    "si-RNA*", "ds-RNA*", 
    "short-interfer* RNA*", "small-interfer* RNA*",
    "RNAi molecule*", "RNA interfer* molecule*",
    "siRNA agent*", "siRNA molecule*",
    "silenc* RNA*",
    "ds-oligonucleotide*", "double stranded oligonucleotide*", "RNAi oligonucleotide*"
]

# Critério 2: Códigos de Classificação (CPC)
# C12N15/113 = Ácidos nucleicos usados para alterar a expressão genética
# A61K31/713 = Preparações médicas contendo ácidos nucleicos de dupla cadeia
cpc_query = '(cpc="C12N15/113" OR cpc="A61K31/713")'

# Período temporal de extração (podes adicionar mais anos a esta lista, ex: [2024, 2025, 2026])
anos_para_pesquisar = [2026]


# ==========================================
# 2. FUNÇÕES AUXILIARES E DE EXTRAÇÃO
# ==========================================

def clean_val(node):
    """
    Limpa os dados recebidos da API. O EPO por vezes devolve o texto dentro 
    de um dicionário com a chave '$'. Esta função extrai apenas o texto útil.
    """
    if isinstance(node, dict):
        return node.get('$', '')
    return str(node) if node else ""

def get_access_token():
    """
    Vai à API do EPO pedir um token temporário para fazer as pesquisas.
    Este token expira em 20 minutos.
    """
    url = "https://ops.epo.org/3.2/auth/accesstoken"
    # Codifica as credenciais no formato exigido pela API (Base64)
    auth = base64.b64encode(f"{CONSUMER_KEY}:{CONSUMER_SECRET}".encode()).decode()
    
    res = requests.post(url,
        headers={'Authorization': f'Basic {auth}', 'Content-Type': 'application/x-www-form-urlencoded'},
        data={'grant_type': 'client_credentials'})
    res.raise_for_status() # Se der erro (ex: credenciais erradas), o programa para aqui
    
    return res.json()['access_token']

def pesquisar_ids(token, cql_query, ano):
    """
    Procura as patentes que correspondem aos nossos critérios e devolve 
    apenas os seus números de identificação (IDs).
    A API só permite descarregar 100 resultados de cada vez, por isso usamos paginação.
    """
    q = urllib.parse.quote(f'{cql_query} AND pd={ano}') # Formata o texto para formato de URL
    ids_extraidos = []
    
    start = 1
    while start <= 2000: # Limite máximo de resultados que o EPO permite por pesquisa
        end = start + 99
        headers = {
            'Authorization': f'Bearer {token}',
            'Accept': 'application/json',
            'X-OPS-Range': f'{start}-{end}' # Pede os resultados da posição 'start' até 'end'
        }
        res = requests.get(f"https://ops.epo.org/3.2/rest-services/published-data/search?q={q}", headers=headers)

        # Se não houver mais resultados (erro 404), sai do loop
        if res.status_code != 200: 
            if res.status_code == 404:
                break
            else:
                print(f"\n[ERRO DA API EPO] O servidor rejeitou o pedido. Código: {res.status_code}")
                print(f"Detalhes: {res.text}")
                break

        # Navega pelo JSON complexo do EPO para chegar à lista de resultados
        data = res.json().get('ops:world-patent-data', {}).get('ops:biblio-search', {})
        docs = data.get('ops:search-result', {}).get('ops:publication-reference', [])
        
        if not docs: 
            break
        if isinstance(docs, dict): # Se for só 1 resultado, transforma em lista para o loop funcionar
            docs = [docs]

        # Extrai o ID formato docdb (País + Número + Tipo) de cada documento
        for doc in docs:
            dids = doc.get('document-id', doc.get('ops:document-id', []))
            if isinstance(dids, dict): 
                dids = [dids]
            
            docdb_node = next((d for d in dids if d.get('@document-id-type') == 'docdb'), None)
            if docdb_node:
                c = clean_val(docdb_node.get('country', ''))
                num = clean_val(docdb_node.get('doc-number', ''))
                k = clean_val(docdb_node.get('kind', ''))
                id_parts = [p for p in [c, num, k] if p]
                if len(id_parts) >= 2:
                    ids_extraidos.append(".".join(id_parts))

        start += 100
        time.sleep(2) # Pausa obrigatória para não sobrecarregar o servidor do EPO (senão bloqueiam o IP)
        
    return ids_extraidos

def obter_detalhes(token, lista_ids):
    """
    Pega na lista de IDs e vai pedir os detalhes completos (Título, Data, etc.) 
    Pede em blocos de 50 para ser mais rápido (Batching).
    """
    print(f"\n[INFO] A iniciar extração de metadados para {len(lista_ids)} patentes...")
    resultados = []
    
    for i in range(0, len(lista_ids), 50):
        lote = lista_ids[i:i+50]
        url = f"https://ops.epo.org/3.2/rest-services/published-data/publication/docdb/{','.join(lote)}/biblio"
        res = requests.get(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/json'})

        if res.status_code == 200:
            _extrair_doc(res.json(), resultados)
        elif res.status_code == 404:
            # Se o pedido em bloco falhar, tenta pedir um por um (Fallback)
            for id_individual in lote:
                url_ind = f"https://ops.epo.org/3.2/rest-services/published-data/publication/docdb/{id_individual}/biblio"
                r = requests.get(url_ind, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/json'})
                if r.status_code == 200: 
                    _extrair_doc(r.json(), resultados)
                time.sleep(0.5)
        time.sleep(3) # Pausa entre blocos para respeitar os limites do servidor
    return resultados

def _extrair_doc(json_data, resultados):
    """
    Função interna que lê a resposta gigante do EPO e guarda apenas 
    os campos que nos interessam num formato limpo.
    """
    data = json_data.get('ops:world-patent-data', {})
    ex_docs = data.get('ops:exchange-documents') or data.get('exchange-documents') or {}
    docs = ex_docs.get('ops:exchange-document') or ex_docs.get('exchange-document') or []
    
    if isinstance(docs, dict): 
        docs = [docs]

    for doc in docs:
        bib = doc.get('ops:bibliographic-data', doc.get('bibliographic-data', {}))
        
        # 1. Extração do Título
        t_node = bib.get('ops:invention-title', bib.get('invention-title', []))
        if isinstance(t_node, dict): 
            t_node = [t_node]
        titulo = clean_val(t_node[0]) if t_node else "Sem título"

        # 2. Extração do País, Número e Data de Publicação
        p_ref = bib.get('ops:publication-reference', bib.get('publication-reference', {}))
        d_ids = p_ref.get('ops:document-id', p_ref.get('document-id', []))
        if isinstance(d_ids, dict): 
            d_ids = [d_ids]
        
        dt, cc, nr, kc = "", "", "", ""
        for d in d_ids:
            if not dt: 
                dt = clean_val(d.get('date', ''))
            if d.get('@document-id-type') == 'docdb':
                cc = clean_val(d.get('country', '')).strip()
                nr = clean_val(d.get('doc-number', '')).replace(".", "").replace("-", "").strip()
                kc = clean_val(d.get('kind', '')).strip()

        # 3. Extração e arrumação dos Códigos de Classificação (CPCs)
        cpcs_set = set() # Usar um 'set' evita que fiquemos com códigos repetidos
        c_wrap = bib.get('ops:patent-classifications', bib.get('patent-classifications', {}))
        c_list = c_wrap.get('ops:patent-classification', c_wrap.get('patent-classification', []))
        if isinstance(c_list, dict): 
            c_list = [c_list]
        
        for c in c_list:
            s = clean_val(c.get('section',''))
            cl = clean_val(c.get('class',''))
            sub = clean_val(c.get('subclass',''))
            main = clean_val(c.get('main-group',''))
            subg = clean_val(c.get('subgroup',''))
            
            # Junta as partes todas para formar o código completo (Ex: C12N15/113)
            full_code = f"{s}{cl}{sub}{main}/{subg}".strip()
            if len(full_code) > 2:
                cpcs_set.add(full_code)

        # Guarda as informações processadas na lista final
        resultados.append({
            'ID_Patente': f"{cc}{nr}{kc}",
            'País': cc, 'Num': nr, 'Tipo': kc,
            'Family_ID': doc.get('@family-id', ''), # ID da família para removermos clones depois
            'Data_Publicacao': dt,
            'Titulo': titulo,
            'CPCs': ", ".join(sorted(list(cpcs_set))) # Junta os códigos separados por vírgula
        })


# ==========================================
# 3. LÓGICA DE EXECUÇÃO PRINCIPAL
# ==========================================

try:
    # 1. Autenticação inicial
    tk = get_access_token()
    
    # 2. A API queixa-se se a pesquisa for muito longa, por isso dividimos
    # a nossa lista de termos em blocos de 4 termos de cada vez.
    tamanho_bloco = 4
    blocos_termos = [termos_rna[i:i + tamanho_bloco] for i in range(0, len(termos_rna), tamanho_bloco)]
    
    ids_totais = []
    
    for a in anos_para_pesquisar:
        print(f"\n[INFO] A processar o ano {a}. Dividindo os {len(termos_rna)} termos em {len(blocos_termos)} pesquisas parciais...")
        
        # Faz uma pesquisa separada para cada bloco de 4 palavras
        for idx, bloco in enumerate(blocos_termos):
            print(f"  -> A executar pesquisa parcial {idx + 1}/{len(blocos_termos)}...")
            
            # Procura os termos no Título/Resumo (txt) OU nas Reivindicações legais (cl)
            txt_query = " OR ".join([f'(txt="{termo}" OR cl="{termo}")' for termo in bloco])
            cql_parcial = f'({txt_query}) AND {cpc_query}'
            
            ids_totais.extend(pesquisar_ids(tk, cql_parcial, a))
            time.sleep(1)
            
    # 3. Limpeza inicial: Remove IDs que possam ter aparecido repetidos nas várias pesquisas
    ids_totais = list(set(ids_totais))

    if not ids_totais:
        print("\n[AVISO] Nenhuma patente encontrada com os parâmetros fornecidos.")
    else:
        # 4. Vai buscar o Título, Data e CPCs de todos os IDs encontrados
        final_data = obter_detalhes(tk, ids_totais)
        
        if final_data:
            # Transforma a lista numa tabela (DataFrame) do Pandas para ser fácil manipular
            df = pd.DataFrame(final_data)
            
            # 5. FILTRO AGRESSIVO DE EXCLUSÃO
            # Garante que eliminamos patentes sobre plantas, agricultura ou insetos.
            # O símbolo '~' significa "Manter tudo o que NÃO contém isto"
            df = df[~df['CPCs'].str.contains('A01H|A01N|A01P|C05|A61M|C12P|C12N5|C12N15/82|C12N15/80|C07K14/4356', na=False)]
            
            # 6. Agrupa por Família de Patente
            # A mesma patente em países diferentes partilha o mesmo Family_ID.